In [1]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
MODEL = "text-embedding-3-small" # 1536차원
# MODEL = "text-embedding-3-large" # 3072차원

In [4]:
from openai import OpenAI

client = OpenAI()

response = client.embeddings.create(
    model=MODEL,
    input="나는 오늘 짜장면을 먹었어",    
)

print(response.data[0].embedding)
print(len(response.data[0].embedding))

[0.03790283203125, -0.022979736328125, -0.0887451171875, 0.005970001220703125, 0.015777587890625, 0.0256195068359375, 0.0343017578125, 0.052886962890625, -0.0197601318359375, -0.001766204833984375, -0.0197906494140625, 0.018585205078125, 0.0017843246459960938, 0.0692138671875, -0.015716552734375, -0.0278778076171875, 0.0162353515625, 0.0257720947265625, 0.015777587890625, 0.033355712890625, 0.0306854248046875, -0.0090179443359375, -0.0036296844482421875, -0.00522613525390625, -0.0231475830078125, 0.03271484375, -0.00307464599609375, -0.0186920166015625, -0.00606536865234375, -0.0217437744140625, -0.0157470703125, -0.0306549072265625, -0.033905029296875, -0.043792724609375, 0.0029277801513671875, -0.01560211181640625, -0.0160980224609375, 0.014923095703125, 0.005825042724609375, 0.031890869140625, -0.01268768310546875, -0.011077880859375, 0.034393310546875, 0.01336669921875, 0.036529541015625, -0.017913818359375, -0.004016876220703125, 0.05181884765625, 0.005725860595703125, -0.02285766

#### 벡터 유사도
- 벡터 숫자 집합이 나타내는 항목이 서로 얼마나 유사한가?

In [3]:
import numpy as np
from numpy import dot
import pandas as pd
from numpy.linalg import norm

In [11]:
def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

In [7]:
vec1 = np.array([0,1,1,1])
vec2 = np.array([1,0,1,1])
vec3 = np.array([2,0,2,2])

print("벡터1과 벡터2 유사도",cosine_similarity(vec1,vec2))
print("벡터1과 벡터3 유사도",cosine_similarity(vec1,vec3))
print("벡터2와 벡터3 유사도",cosine_similarity(vec2,vec3))

벡터1과 벡터2 유사도 0.6666666666666667
벡터1과 벡터3 유사도 0.6666666666666667
벡터2와 벡터3 유사도 1.0000000000000002


In [8]:
data = [
    '저는 배가 고파요',
    '저기 배가 지나가네요',
    '굶어서 허기가 지네요',
    '허기 워기라는 게임이 있는데 즐거워',
    '스팀에서 재밌는거 해야지',
    '스팀에어프라이어로 연어구이 해 먹을거야',
]

df = pd.DataFrame(data, columns=['text'])
df

,text
0,저는 배가 고파요
1,저기 배가 지나가네요
2,굶어서 허기가 지네요
3,허기 워기라는 게임이 있는데 즐거워
4,스팀에서 재밌는거 해야지
5,스팀에어프라이어로 연어구이 해 먹을거야


In [9]:
def get_embeddings(text):
    response = client.embeddings.create(
        model=MODEL,
        input=text,    
    )
    return response.data[0].embedding

In [10]:
# embedding 컬럼 추가

df['embedding'] = df.apply(lambda row:get_embeddings(row.text), axis=1)
df

,text,embedding
0,저는 배가 고파요,"[0.017365, -0.007858, -0.029099, 0.007553, 0.0..."
1,저기 배가 지나가네요,"[0.03949, 0.000087, -0.014183, 0.003641, -0.02..."
2,굶어서 허기가 지네요,"[0.041534, 0.024826, -0.012794, 0.003227, 0.00..."
3,허기 워기라는 게임이 있는데 즐거워,"[0.007484, -0.003967, -0.065552, -0.039795, 0...."
4,스팀에서 재밌는거 해야지,"[0.00721, 0.054016, -0.041931, 0.007629, 0.009..."
5,스팀에어프라이어로 연어구이 해 먹을거야,"[0.048645, -0.003414, 0.00002, -0.020737, 0.00..."


In [11]:
# 사용자의 문장과 유사성이 가장 높은 3문장 추출

def return_answer_candidate(df, query):
    # 질문 벡터 변환
    query_embedding = get_embeddings(query)
    # query 문장과 비교하기 위해 유사도 컬럼 생성
    df['similarity'] = df.embedding.apply(lambda x:cosine_similarity(np.array(x), np.array(query_embedding)))

    # top3 추출
    top_three = df.sort_values("similarity", ascending=False).head(3)
    return top_three

In [12]:
sim_result = return_answer_candidate(df, "배에서 꼬르륵 소리가 나네")
sim_result

,text,embedding,similarity
1,저기 배가 지나가네요,"[0.03949, 0.000087, -0.014183, 0.003641, -0.02...",0.311706
4,스팀에서 재밌는거 해야지,"[0.00721, 0.054016, -0.041931, 0.007629, 0.009...",0.276267
2,굶어서 허기가 지네요,"[0.041534, 0.024826, -0.012794, 0.003227, 0.00...",0.272577


In [13]:
documents = [
    "강아지는 사람과 친근하게 지내는 동물이다.",
    "고양이는 독립적인 성격을 가진 동물이다.",
    "인공지능은 인간의 지능을 모방하는 기술이다.",
    "머신러닝은 데이터를 통해 패턴을 학습하는 기술이다.",
    "오늘 저녁에는 피자를 먹었다.",
]

df = pd.DataFrame(documents, columns=['text'])
df['embedding'] = df.text.apply(lambda t:get_embeddings(t))
df

,text,embedding
0,강아지는 사람과 친근하게 지내는 동물이다.,"[0.00885, 0.008057, -0.032135, 0.016632, 0.018..."
1,고양이는 독립적인 성격을 가진 동물이다.,"[0.032745, 0.027878, -0.026276, 0.036102, 0.01..."
2,인공지능은 인간의 지능을 모방하는 기술이다.,"[0.004562, 0.056213, -0.004696, 0.017349, -0.0..."
3,머신러닝은 데이터를 통해 패턴을 학습하는 기술이다.,"[-0.027008, 0.016968, 0.012413, 0.010559, 0.07..."
4,오늘 저녁에는 피자를 먹었다.,"[-0.021103, -0.043976, -0.087646, -0.051758, 0..."


In [14]:
sim_result = return_answer_candidate(df,"개는 어떤 특징을 가진 동물인가요?")
sim_result

,text,embedding,similarity
1,고양이는 독립적인 성격을 가진 동물이다.,"[0.032745, 0.027878, -0.026276, 0.036102, 0.01...",0.399442
0,강아지는 사람과 친근하게 지내는 동물이다.,"[0.00885, 0.008057, -0.032135, 0.016632, 0.018...",0.352250
2,인공지능은 인간의 지능을 모방하는 기술이다.,"[0.004562, 0.056213, -0.004696, 0.017349, -0.0...",0.212645


### 아마존 고급 음식 리뷰
- https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews

In [10]:
# csv 로드

food_df = pd.read_csv("./data/fine_food_reviews_1k_fully_translated_korean.csv", index_col=0)
food_df.head()

,Time,ProductId,UserId,Score,Summary,Text
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,조각으로 도착했습니다,상자를 열었을 때 전혀 기뻐하지 않습니다. 대부분의 고리가 조각으로 부러졌습니다.
2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,그것은 Blanc Mange는 아니지만 나쁘지 않습니다,커스터드가 계란이없는 커스터드인지 확실하지 않지만 이것은 비건 채식 팬케이크 레시피...
3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,이들은 또한 소금과 바다 소금이 아닙니다,나는 당신이 당신이 무엇을 얻는 지 알 수 있고 뼈 나 어두운 고기가 없다는 사실을...
4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,제품에 만족합니다,내 개는 가려운 피부로 고통 스러웠습니다.음식과 피부 문제는 몇 주 안에 정리되었습니다.


In [17]:
# text + Summary 결합 컬럼 생성
food_df['Combined'] = "Title: "+food_df.Summary.str.strip() + "; Content: " + food_df.Text.str.strip()

In [18]:
# time 컬럼 제거
food_df = food_df.drop(columns=['Time'])
food_df.head(1)

,ProductId,UserId,Score,Summary,Text,Combined
0,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...,Title: 이런 식으로 어디서 시작하고 멈추는가; Content: 시카고 가족에게...


In [19]:
food_df.shape

(999, 6)

In [20]:
# 임베딩 값 추출

food_df['embedding'] = food_df.Combined.apply(lambda t:get_embeddings(t))
food_df.head(2)

,ProductId,UserId,Score,Summary,Text,Combined,embedding
0,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...,Title: 이런 식으로 어디서 시작하고 멈추는가; Content: 시카고 가족에게...,"[-0.00239, 0.000758, -0.019791, 0.044556, 0.00..."
1,B003JK537S,A3JBPC3WFUT5ZP,1,조각으로 도착했습니다,상자를 열었을 때 전혀 기뻐하지 않습니다. 대부분의 고리가 조각으로 부러졌습니다.,Title: 조각으로 도착했습니다; Content: 상자를 열었을 때 전혀 기뻐하지...,"[0.057068, 0.005138, -0.055054, 0.002146, 0.04..."


In [21]:
food_df.to_csv("data/food_df.csv")

In [31]:
from openai import OpenAI
import pandas as pd 

df = pd.read_csv("./data/food_df.csv" , index_col=0 )

df['embedding']=df.embedding.apply(eval)
client = OpenAI()
df.head(2)

,ProductId,UserId,Score,Summary,Text,Combined,embedding
0,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...,Title: 이런 식으로 어디서 시작하고 멈추는가; Content: 시카고 가족에게...,"[-0.00239, 0.000758, -0.019791, 0.044556, 0.00..."
1,B003JK537S,A3JBPC3WFUT5ZP,1,조각으로 도착했습니다,상자를 열었을 때 전혀 기뻐하지 않습니다. 대부분의 고리가 조각으로 부러졌습니다.,Title: 조각으로 도착했습니다; Content: 상자를 열었을 때 전혀 기뻐하지...,"[0.057068, 0.005138, -0.055054, 0.002146, 0.04..."


In [32]:
def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

In [33]:

def get_embeddings(text):
    response = client.embeddings.create(
        model=MODEL,
        input=text,    
    )
    return response.data[0].embedding

In [34]:
# 사용자의 문장과 유사성이 가장 높은 3문장 추출

def search_reviews(df, query, n=3):
    # 질문 벡터 변환
    query_embedding = get_embeddings(query)
    # query 문장과 비교하기 위해 유사도 컬럼 생성
    df['similarity'] = df.embedding.apply(lambda x:cosine_similarity(np.array(x), np.array(query_embedding)))

    res = df.sort_values("similarity", ascending=False).head(n)
    return res

In [35]:
# 유사한 리뷰 찾기
# 질문 => 임베딩 

search_reviews(df, '맛있는 콩', 4)

,ProductId,UserId,Score,Summary,Text,Combined,embedding,similarity
771,B002GWHBU2,A2NUQSB42RKBFA,5,자메이카 푸른 콩,로스팅하기위한 우수한 커피 콩 우리 가족은 더 많은 양의 맛을 위해 5 파운드를 더...,Title: 자메이카 푸른 콩; Content: 로스팅하기위한 우수한 커피 콩 우리...,"[0.003052, 0.039215, -0.053925, -0.009491, 0.0...",0.519728
88,B000E3ZFDU,A1PQDL14230X6U,5,맛있는,나는이 흰 콩 조미료를 즐긴다 그것은 콩에 풍부한 맛을줍니다. 나는 법에있는 어머니...,Title: 맛있는; Content: 나는이 흰 콩 조미료를 즐긴다 그것은 콩에 풍...,"[0.0466, 0.034332, -0.074524, -0.004642, 0.021...",0.519335
460,B000RZNY5G,A2ZGTXBI8C7FYI,5,아주 맛있는,아주 좋은 것은 여전히 ​​약간 왼쪽을 가지고 있습니다. 나는 짠 맛이 마음에 든다.,Title: 아주 맛있는; Content: 아주 좋은 것은 여전히 ​​약간 왼쪽을 ...,"[-0.009079, -0.031158, -0.073669, 0.00135, 0.0...",0.484991
60,B000LVM4EI,ATTK7J5JWVDYD,5,아주 맛있는,최고의 matzo balls 거기에 너무 맛있는 소금으로 너무 맛있어 너무 나쁘다.,Title: 아주 맛있는; Content: 최고의 matzo balls 거기에 너무...,"[0.011635, 0.006687, -0.060608, 0.026016, 0.03...",0.464602


In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 999 entries, 0 to 999
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   ProductId  999 non-null    str  
 1   UserId     999 non-null    str  
 2   Score      999 non-null    int64
 3   Summary    999 non-null    str  
 4   Text       999 non-null    str  
 5   Combined   999 non-null    str  
 6   embedding  999 non-null    str  
dtypes: int64(1), str(6)
memory usage: 62.4 KB


#### FAISS
- Facebook AI Similarity Search
- 벡터 검색 라이브러리 ( 대량의 벡터 데이터에서 유사한 데이터를 빠르고 효율적으로 검색하기 위한 도구 )
- 인덱스 검색
    - IndexFlatL2 : 모든 벡터를 비교하는 검색
    - IndexFlatLIP : 내적을 이용한 검색
    - ...

In [ ]:
#uv add faiss-cpu

In [37]:
# 임베딩을 하나의 행렬로 만들기

print("리뷰개수",
len(df))
print(len(df['embedding'].iloc[0]))

vectors = np.array(df['embedding'].tolist(),dtype="float32")
vectors.shape

리뷰개수 999
1536


(999, 1536)

In [38]:
# 정규화

def to_unit_length(v):
    """벡터 길이 1로 만들기"""
    length = np.linalg.norm(v, axis=1, keepdims=True)
    return v /length

vectors = to_unit_length(vectors)

In [ ]:
import faiss
#
dim = vectors.shape[1]
index=faiss.IndexFlatIP(dim) # 내적 이용 검색
index.add(vectors)


In [43]:

print("인덱스에 저장된 벡터 수 ", index.ntotal)

인덱스에 저장된 벡터 수  999


In [44]:
query_embedding = get_embeddings("맛있는 콩")
query_vec = np.array([query_embedding], dtype="float32")
query_vec = to_unit_length(query_vec)

print(query_vec.shape)

(1, 1536)


In [45]:
scores,positioins = index.search(query_vec, 4)

print("유사도 점수 ",scores[0])
print("찾은 인덱스 ",positioins[0])

유사도 점수  [0.5197134  0.5192672  0.48491472 0.46443033]
찾은 인덱스  [770  88 460  60]


In [47]:
result = df.iloc[positioins[0]].copy()

result['sim'] = scores[0]
result[['Summary','Text','sim']]

,Summary,Text,sim
771,자메이카 푸른 콩,로스팅하기위한 우수한 커피 콩 우리 가족은 더 많은 양의 맛을 위해 5 파운드를 더...,0.519713
88,맛있는,나는이 흰 콩 조미료를 즐긴다 그것은 콩에 풍부한 맛을줍니다. 나는 법에있는 어머니...,0.519267
460,아주 맛있는,아주 좋은 것은 여전히 ​​약간 왼쪽을 가지고 있습니다. 나는 짠 맛이 마음에 든다.,0.484915
60,아주 맛있는,최고의 matzo balls 거기에 너무 맛있는 소금으로 너무 맛있어 너무 나쁘다.,0.464430
